# Restaurant Annual Turnover — Improvement 6

**Student:** submission · **Register number:** submission · **Team:** submission

Prepared 26 October 2026. In Colab select a CPU runtime and run all cells. Upload
the four official CSVs when prompted. Save the executed notebook using File → Download.

**Measured decision: retain portal-scored v5.** Selected predictor: `v5`;
new-model weight: 0%. The frozen v5 portal score is
**12,046,722.3691139**, as confirmed by the user. The shown leader is **4,796,740.5600053**.
This notebook does not claim an unobserved portal score or rank.

## Runtime and input files

This notebook embeds all predictor source code. It does not require older improvement
folders, prediction caches, local helper scripts or a Downloads directory. It installs
the measured package versions when necessary. Fresh fitting is performed below.

In [ ]:
import os, sys, json, hashlib, base64, zlib, types, subprocess, importlib.metadata
from pathlib import Path

PACKAGES = {'numpy': '2.5.4', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'catboost': '1.2.10', 'lightgbm': '4.7.0', 'scipy': '1.18.1'}
to_install = []
for package, version in PACKAGES.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        to_install.append(f'{package}=={version}')
if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
try:
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'])
    import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
print('Python:', sys.version.split()[0])
print('Packages:', {name: importlib.metadata.version(name) for name in PACKAGES})

In [ ]:
REQUIRED = ['Train_dataset.csv', 'Test_dataset.csv', 'Data_Dictionary.csv', 'Sample_Solution.csv']
DATA_DIR = Path(os.environ.get('MLA_DATA_DIR', '.')).resolve()
OUTPUT_DIR = Path(os.environ.get('MLA_OUTPUT_DIR', '.')).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not all((DATA_DIR / name).is_file() for name in REQUIRED):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place the four official CSVs in the current folder or set MLA_DATA_DIR.')
    files.upload()
    DATA_DIR = Path.cwd()
assert all((DATA_DIR / name).is_file() for name in REQUIRED)
INPUT_HASHES = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in REQUIRED}

EXPECTED_INPUTS = {'Train_dataset.csv': 'ec40ebda74ba0bfbfe1af4417a4266a70eadf6e595500e99e5435e7bdc7ee51f', 'Test_dataset.csv': 'ee4137566adb1c0ce356a8c370e7b8a22f681bde492068efe11346d8d44b4f4d', 'Data_Dictionary.csv': '2d86c00e27448593564a161197b6639ee1eb90c01fa523b67e7a09d2ef405c5e', 'Sample_Solution.csv': '9519ba4d51eb2d59b8a71b054a62e1f4d48b72cf92930c31f5e5393a2cd135fb'}
assert INPUT_HASHES == EXPECTED_INPUTS, 'These are not the four measured competition inputs.'
print('PASS: all four input SHA256 hashes match the measured experiment.')

## Frozen reference and six bounded hypotheses

The following source is embedded from the audited implementation. The v5 module is
isolated to keep its legacy functions separate from new predictor functions.
The six settings use native-categorical LightGBM (L2/Huber, 7/15 leaves), a square-root
CatBoost model corrected using three inner OOF folds, and bounded feature-based
distribution weights for Huber CatBoost. All scores retain original INR targets.

In [ ]:
REFERENCE_SOURCE = '"""Frozen v5 predictor, copied verbatim from the preserved implementations."""\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor\nfrom sklearn.model_selection import KFold, GroupKFold\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor\nfrom sklearn.svm import SVR\n\n\nTARGET = \'Annual Turnover\'\n\nID = \'Registration Number\'\n\nEVENTS = [\'Live Music Rating\',\'Comedy Gigs Rating\',\'Value Deals Rating\',\'Live Sports Rating\']\n\nSOCIAL = [\'Facebook Popularity Quotient\',\'Instagram Popularity Quotient\']\n\nCORE = SOCIAL + [\'Hygiene Rating\',\'Restaurant Zomato Rating\',\'Live Music Rating\',\'Value for Money\',\'Overall Restaurant Rating\',\'Restaurant City Tier\',\'Cuisine\',\'Restaurant Theme\',\'Restaurant Location\',\'Endoresed By\',\'City\',\'Opening Year\']\n\ndef features(raw, variant=\'base\'):\n    data=raw.drop(columns=[TARGET,ID],errors=\'ignore\').copy()\n    dates=pd.to_datetime(data.pop(\'Opening Day of Restaurant\'),format=\'%d-%m-%Y\',errors=\'coerce\')\n    data[\'Opening Year\']=dates.dt.year.astype(float)\n    data[\'Opening Month\']=dates.dt.month.astype(float)\n    for c in EVENTS:\n        data[\'Offers \'+c.replace(\' Rating\',\'\')]=data[c].notna().astype(int)\n    parts=data[\'Cuisine\'].astype(\'string\').fillna(\'Missing\').str.split(\',\',n=1,expand=True)\n    data[\'Cuisine 1\']=parts[0].str.strip().str.lower()\n    data[\'Cuisine 2\']=parts.reindex(columns=[0,1])[1].astype(\'string\').str.strip().str.lower()\n    for c in [\'Cuisine\',\'City\',\'Restaurant Location\',\'Endoresed By\',\'Restaurant Type\',\'Restaurant Theme\']:\n        data[c]=data[c].astype(\'string\').str.strip().str.lower().str.replace(r\'\\s+\',\' \',regex=True).fillna(\'Missing\')\n    data[\'City Is Unknown\']=(data[\'City\']==\'-1\').astype(int)\n    for c in data.select_dtypes(include=[\'object\',\'string\']).columns:\n        data[c]=data[c].fillna(\'Missing\').astype(str)\n    if variant in [\'engineered\',\'engineered_drop\']:\n        data[\'City\']=data[\'City\'].replace({\'bengaluru\':\'bangalore\',\'gurugram\':\'gurgaon\',\'bombay\':\'mumbai\',\'calcutta\':\'kolkata\',\'delhi\':\'new delhi\'})\n        data[\'Cuisine unordered\']=[\'|\'.join(sorted(v.split(\',\'))) for v in data[\'Cuisine\']]\n        data[\'Opening Day\']=dates.dt.day.astype(float)\n        data[\'Opening Days Since 1970\']=(dates-pd.Timestamp(\'1970-01-01\')).dt.days.astype(float)\n        data[\'Social mean\']=data[SOCIAL].mean(axis=1)\n        data[\'Social difference\']=data[SOCIAL[0]]-data[SOCIAL[1]]\n        data[\'Social missing count\']=data[SOCIAL].isna().sum(axis=1)\n        data[\'Rating missing count\']=data.isna().sum(axis=1)\n        data[\'Event count\']=data[EVENTS].notna().sum(axis=1)\n        data[\'Event mean\']=data[EVENTS].mean(axis=1)\n        data[\'Quality mean\']=data[[\'Food Rating\',\'Hygiene Rating\',\'Overall Restaurant Rating\']].mean(axis=1)\n        data[\'Social x Zomato\']=data[\'Social mean\']*data[\'Restaurant Zomato Rating\']\n        data[\'Social x Hygiene\']=data[\'Social mean\']*data[\'Hygiene Rating\']\n        data[\'Quality x Value\']=data[\'Quality mean\']*data[\'Value for Money\']\n    if variant in [\'drop_shift\',\'engineered_drop\']:\n        data=data.drop(columns=[\'Service\',\'Comfortablility\'])\n    if variant==\'core\': data=data[CORE]\n    if variant==\'ratings\':\n        keep=SOCIAL+[\'Resturant Tier\',\'Restaurant Zomato Rating\',\'Restaurant City Tier\',\'Staff Responsivness\',\'Value for Money\',\'Hygiene Rating\',\'Overall Restaurant Rating\']+EVENTS+[\'Lively\',\'Ambience\',\'Privacy\']\n        data=data[keep]\n    if variant==\'no_city_date\':\n        data=data.drop(columns=[\'City\',\'City Is Unknown\',\'Opening Year\',\'Opening Month\'])\n    return data\n\ndef rmse(y,p): return float(np.sqrt(np.mean(np.square(np.asarray(y)-np.asarray(p)))))\n\ndef sklearn_model(X,spec,seed):\n    cats=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist()\n    nums=[c for c in X if c not in cats]\n    num=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True),StandardScaler())\n    if spec[\'kind\']==\'spline\':\n        num=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True),SplineTransformer(n_knots=4,degree=2,include_bias=False),StandardScaler())\n    pre=ColumnTransformer([(\'num\',num,nums),(\'cat\',OneHotEncoder(handle_unknown=\'ignore\',min_frequency=10,sparse_output=False),cats)])\n    kind=spec[\'kind\']\n    if kind in [\'ridge\',\'spline\']: model=Ridge(alpha=spec.get(\'alpha\',100))\n    elif kind==\'svr\': model=SVR(C=spec.get(\'C\',20),epsilon=.5,kernel=\'rbf\')\n    elif kind==\'hist\': model=HistGradientBoostingRegressor(max_iter=300,max_leaf_nodes=10,max_depth=4,learning_rate=.035,l2_regularization=30,min_samples_leaf=60,random_state=seed)\n    elif kind==\'extra\': model=ExtraTreesRegressor(n_estimators=400,min_samples_leaf=5,max_features=.7,n_jobs=4,random_state=seed)\n    return make_pipeline(pre,model)\n\ndef fit_predict(X,y,train_idx,valid_idx,spec,seed,fixed_iterations=None,return_fitted=False):\n    ys=np.asarray(y)[train_idx].copy()\n    if \'cap\' in spec: ys=np.minimum(ys,spec[\'cap\'])\n    if spec[\'kind\']==\'cat\':\n        params=dict(loss_function=spec.get(\'loss\',\'RMSE\'),eval_metric=\'RMSE\',iterations=fixed_iterations or spec.get(\'iterations\',1600),learning_rate=.04,depth=spec.get(\'depth\',6),l2_leaf_reg=spec.get(\'l2\',5),random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False)\n        if \'boosting_type\' in spec: params[\'boosting_type\']=spec[\'boosting_type\']\n        if spec.get(\'monotone\'): params[\'monotone_constraints\']={**{c:1 for c in SOCIAL+[\'Hygiene Rating\',\'Restaurant Zomato Rating\']},\'Resturant Tier\':-1}\n        model=CatBoostRegressor(**params)\n        cat=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist()\n        stopping={} if fixed_iterations or \'iterations\' in spec else dict(eval_set=(X.iloc[valid_idx],np.asarray(y)[valid_idx]),early_stopping_rounds=120)\n        model.fit(X.iloc[train_idx],ys,cat_features=cat,**stopping)\n        p=model.predict(X.iloc[valid_idx]); trees=model.tree_count_\n    elif spec[\'kind\'] in [\'lgb\',\'xgb\']:\n        pre=sklearn_model(X,dict(kind=\'ridge\'),seed)[:-1]\n        ztr=pre.fit_transform(X.iloc[train_idx]); zva=pre.transform(X.iloc[valid_idx])\n        early=not (fixed_iterations or \'iterations\' in spec)\n        iterations=fixed_iterations or spec.get(\'iterations\',1400)\n        if spec[\'kind\']==\'lgb\':\n            from lightgbm import LGBMRegressor,early_stopping\n            model=LGBMRegressor(n_estimators=iterations,learning_rate=.025,num_leaves=spec.get(\'leaves\',15),min_child_samples=spec.get(\'leaf_samples\',60),reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,subsample_freq=1,random_state=seed,n_jobs=4,verbosity=-1,objective=spec.get(\'objective\',\'regression\'))\n            params=dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],eval_metric=\'rmse\',callbacks=[early_stopping(100,verbose=False,first_metric_only=True)]) if early else {}\n            model.fit(ztr,ys/1e6,**params)\n            trees=model.best_iteration_ if early else iterations\n        else:\n            from xgboost import XGBRegressor\n            params=dict(n_estimators=iterations,max_depth=spec.get(\'depth\',3),learning_rate=.025,min_child_weight=30,reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,random_state=seed,n_jobs=4,tree_method=\'hist\',objective=spec.get(\'objective\',\'reg:squarederror\'),eval_metric=\'rmse\',base_score=float(ys.mean()/1e6))\n            if early: params[\'early_stopping_rounds\']=100\n            if spec.get(\'objective\')==\'reg:pseudohubererror\': params[\'huber_slope\']=12\n            model=XGBRegressor(**params)\n            model.fit(ztr,ys/1e6,**(dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],verbose=False) if early else {}))\n            trees=model.best_iteration+1 if early else iterations\n        p=model.predict(zva)*1e6\n    else:\n        model=sklearn_model(X,spec,seed)\n        model.fit(X.iloc[train_idx],ys/1e6)\n        p=model.predict(X.iloc[valid_idx])*1e6; trees=0\n    if return_fitted:\n        fitted=dict(model=model,preprocessor=pre if spec[\'kind\'] in [\'lgb\',\'xgb\'] else None,scale=1 if spec[\'kind\']==\'cat\' else 1e6)\n        return np.maximum(p,0),trees,fitted\n    return np.maximum(p,0),trees\n\ndef predict_fitted(fitted,X):\n    transformed=fitted[\'preprocessor\'].transform(X) if fitted[\'preprocessor\'] is not None else X\n    return np.maximum(fitted[\'model\'].predict(transformed)*fitted[\'scale\'],0)\n\ndef cat_fit(X, y, spec, seed):\n    params=dict(iterations=spec[\'iterations\'], learning_rate=.04, depth=spec[\'depth\'],\n                l2_leaf_reg=spec.get(\'l2\',5), random_seed=seed, thread_count=2,\n                verbose=False, allow_writing_files=False,\n                loss_function=f"Huber:delta={spec[\'delta\']*1e6:g}" if \'delta\' in spec else \'RMSE\')\n    if spec.get(\'monotone\'):\n        params[\'monotone_constraints\']={**{c:1 for c in SOCIAL+[\'Hygiene Rating\',\'Restaurant Zomato Rating\']},\'Resturant Tier\':-1}\n    if spec.get(\'ordered\'): params[\'boosting_type\']=\'Ordered\'\n    if \'subsample\' in spec: params.update(bootstrap_type=\'Bernoulli\',subsample=spec[\'subsample\'])\n    if \'random_strength\' in spec: params[\'random_strength\']=spec[\'random_strength\']\n    fitted=CatBoostRegressor(**params)\n    target=np.minimum(y,spec[\'cap\']*1e6) if \'cap\' in spec else y\n    fitted.fit(X,target,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\n    return fitted\n\ndef get_splits(train,scheme):\n    if str(scheme)==\'city\':\n        group=features(train,\'engineered\')[\'City\']\n        splits=list(GroupKFold(5).split(train,groups=group))\n        for tr,va in splits:assert set(group.iloc[tr]).isdisjoint(group.iloc[va])\n    else:splits=list(KFold(5,shuffle=True,random_state=int(scheme)).split(train))\n    counts=np.zeros(len(train),int)\n    for tr,va in splits:\n        assert set(tr).isdisjoint(va);counts[va]+=1\n    assert (counts==1).all()\n    return splits\n\ndef compute_v3(train,y,scheme,contract):\n    splits=get_splits(train,scheme);pred=np.zeros(len(y))\n    for name,spec in contract[\'v2_configs\'].items():\n        if name not in contract[\'v2_weights\']:continue\n        X=features(train,spec[\'variant\']);component=np.zeros(len(y))\n        for f,(tr,va) in enumerate(splits):\n            component[va],_=fit_predict(X,y,tr,va,spec,43+f,spec.get(\'iterations\'))\n        pred+=.5*contract[\'v2_weights\'][name]*component\n    X=features(train,\'base\');spec=contract[\'v3_parameters\']\n    for f,(tr,va) in enumerate(splits):\n        parts=[]\n        for seed in contract[\'v3_seeds\']:\n            model=cat_fit(X.iloc[tr],y[tr],spec,seed+f)\n            parts.append(np.maximum(model.predict(X.iloc[va]),0))\n        pred[va]+=.5*np.mean(parts,axis=0)\n    return pred\n\ndef check_submission(frame,test):\n    assert frame.shape==(500,2),\'Expected 500 rows and two columns\'\n    assert frame.columns.tolist()==[\'Registration Number\',\'Annual Turnover\'],\'Incorrect header\'\n    assert frame[\'Registration Number\'].equals(test[\'Registration Number\']),\'Restaurant IDs/order changed\'\n    assert frame[\'Registration Number\'].is_unique,\'Duplicate restaurant IDs\'\n    assert np.isfinite(frame[\'Annual Turnover\']).all(),\'Non-finite prediction\'\n    assert (frame[\'Annual Turnover\']>=0).all(),\'Negative turnover\'\n\ndef v5_features(raw,variant=\'base\'):\n    X=features(raw,\'engineered\' if variant==\'rich\' else \'base\')\n    if variant in [\'clean\',\'rich\']:\n        X[\'City\']=X[\'City\'].replace({\'bengaluru\':\'bangalore\',\'banglore\':\'bangalore\',\'gurugram\':\'gurgaon\',\'gurgoan\':\'gurgaon\',\'bombay\':\'mumbai\',\'calcutta\':\'kolkata\',\'delhi\':\'new delhi\',\'bhubneshwar\':\'bhubaneswar\',\'hderabad\':\'hyderabad\',\'punr\':\'pune\'})\n    if variant==\'rich\':\n        fb=X[\'Facebook Popularity Quotient\'];ig=X[\'Instagram Popularity Quotient\']\n        X[\'Social ratio\']=fb/(ig+1)\n        X[\'Social product\']=fb*ig\n        X[\'Social minimum\']=np.minimum(fb,ig)\n        X[\'Social absolute gap\']=np.abs(fb-ig)\n        X[\'Quality spread\']=X[[\'Food Rating\',\'Hygiene Rating\',\'Overall Restaurant Rating\']].std(axis=1)\n        X[\'Zomato x Hygiene\']=X[\'Restaurant Zomato Rating\']*X[\'Hygiene Rating\']\n        X[\'City tier x Restaurant tier\']=X[\'Restaurant City Tier\']*X[\'Resturant Tier\']\n    return X\n\ndef inverse_target(z,power):\n    if power==0:return np.expm1(np.minimum(z,20))\n    return np.maximum(z,0)**(1/power)\n\ndef fit_v5(raw,y,spec,seed):\n    X=v5_features(raw,spec.get(\'variant\',\'base\'));target=np.asarray(y,dtype=float)/1e6\n    if \'cap\' in spec:target=np.minimum(target,spec[\'cap\'])\n    power=spec.get(\'power\',1.)\n    transformed=np.log1p(target) if power==0 else target**power\n    loss=spec.get(\'loss\',\'RMSE\')\n    params=dict(iterations=spec.get(\'iterations\',850),learning_rate=.04,depth=spec.get(\'depth\',5),\n                l2_leaf_reg=10,bootstrap_type=\'Bernoulli\',subsample=.7,loss_function=loss,\n                random_seed=seed,thread_count=2,verbose=False,allow_writing_files=False)\n    if \'rsm\' in spec:params[\'rsm\']=spec[\'rsm\']\n    if \'topk\' in spec:\n        pilot=CatBoostRegressor(**{**params,\'iterations\':300})\n        pilot.fit(X,transformed,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\n        ranked=np.argsort(-pilot.feature_importances_);selected=[X.columns[i] for i in ranked[:spec[\'topk\']]]\n        X=X[selected]\n    model=CatBoostRegressor(**params)\n    model.fit(X,transformed,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\n    correction=0.\n    if spec.get(\'correct_mean\'):\n        # Training-only retransformation correction; all validation labels remain unseen.\n        fitted_original=inverse_target(model.predict(X),power)\n        correction=float(np.mean(target-fitted_original))\n    return {\'model\':model,\'columns\':X.columns.tolist(),\'power\':power,\'correction\':correction,\'spec\':spec}\n\ndef predict_v5(fitted,raw):\n    X=v5_features(raw,fitted[\'spec\'].get(\'variant\',\'base\'))[fitted[\'columns\']]\n    return np.maximum(0,(inverse_target(fitted[\'model\'].predict(X),fitted[\'power\'])+fitted[\'correction\'])*1e6)\n\ndef evaluate(train,y,scheme,contract,previous=None):\n    if previous is None:previous=compute_v3(train,y,scheme,contract)\n    prediction=contract[\'v3_weight\']*previous\n    parts={}\n    for name,c in contract[\'components\'].items():\n        p=np.zeros(len(y))\n        for f,(tr,va) in enumerate(get_splits(train,scheme)):\n            one=[]\n            for seed in contract[\'seeds\']:\n                fitted=fit_v5(train.iloc[tr],y[tr],c[\'spec\'],seed+f)\n                one.append(predict_v5(fitted,train.iloc[va]))\n            p[va]=np.mean(one,axis=0)\n        parts[name]=p;prediction+=c[\'weight\']*p\n    return {\'previous\':previous,\'candidate\':prediction,**parts}\n\ndef full_fit(train,test,y,contract):\n    prior=np.zeros(len(test));fitted={\'v2\':{},\'v3\':[],\'new\':{}}\n    tr=np.arange(len(train));va=np.arange(len(train),len(train)+len(test));yy=np.r_[y,np.full(len(test),np.nan)]\n    for name,w in contract[\'v2_weights\'].items():\n        spec=contract[\'v2_configs\'][name]\n        X=pd.concat([features(train,spec[\'variant\']),features(test,spec[\'variant\'])],ignore_index=True)\n        p,_,model=fit_predict(X,yy,tr,va,spec,42,spec.get(\'iterations\'),return_fitted=True)\n        fitted[\'v2\'][name]=model;prior+=.5*w*p\n    for seed in contract[\'v3_seeds\']:\n        model=cat_fit(features(train,\'base\'),y,contract[\'v3_parameters\'],seed);fitted[\'v3\'].append(model)\n        prior+=.5*np.maximum(model.predict(features(test,\'base\')),0)/len(contract[\'v3_seeds\'])\n    candidate=contract[\'v3_weight\']*prior\n    for name,c in contract[\'components\'].items():\n        fitted[\'new\'][name]=[]\n        for seed in contract[\'seeds\']:\n            model=fit_v5(train,y,c[\'spec\'],seed);fitted[\'new\'][name].append(model)\n            candidate+=c[\'weight\']*predict_v5(model,test)/len(contract[\'seeds\'])\n    return prior,candidate,fitted\n\ndef predict_final(raw,fitted,contract):\n    prior=sum(.5*w*predict_fitted(fitted[\'v2\'][name],features(raw,contract[\'v2_configs\'][name][\'variant\'])) for name,w in contract[\'v2_weights\'].items())\n    prior+=.5*np.mean([np.maximum(m.predict(features(raw,\'base\')),0) for m in fitted[\'v3\']],axis=0)\n    p=contract[\'v3_weight\']*prior\n    for name,c in contract[\'components\'].items():p+=c[\'weight\']*np.mean([predict_v5(m,raw) for m in fitted[\'new\'][name]],axis=0)\n    return p\n'
ref = types.ModuleType('reference')
exec(compile(REFERENCE_SOURCE, '<embedded frozen v5>', 'exec'), ref.__dict__)
sys.modules['reference'] = ref
print('Frozen reference source SHA256:', hashlib.sha256(REFERENCE_SOURCE.encode()).hexdigest())

In [ ]:
"""Six predeclared experiments. Targets are in millions internally, INR at I/O."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, train_test_split
import reference as ref

REQUIRED = ['Train_dataset.csv', 'Test_dataset.csv', 'Data_Dictionary.csv', 'Sample_Solution.csv']
SEEDS = [10242, 20242]
SPECS = {
    'native_l2_7': {'family': 'native_l2', 'kind': 'lgb', 'objective': 'regression', 'leaves': 7},
    'native_l2_15': {'family': 'native_l2', 'kind': 'lgb', 'objective': 'regression', 'leaves': 15},
    'native_huber_7': {'family': 'native_huber', 'kind': 'lgb', 'objective': 'huber', 'leaves': 7},
    'native_huber_15': {'family': 'native_huber', 'kind': 'lgb', 'objective': 'huber', 'leaves': 15},
    'sqrt_crossfit': {'family': 'sqrt_crossfit', 'kind': 'sqrt'},
    'domain_huber': {'family': 'domain_huber', 'kind': 'weighted'},
}


def load_data(data_dir):
    directory = Path(data_dir)
    for name in REQUIRED:
        if not (directory / name).is_file():
            raise FileNotFoundError(f'Missing {directory / name}; use --data-dir')
    train = pd.read_csv(directory / REQUIRED[0]).rename(columns={'Endorsed By': 'Endoresed By'})
    test = pd.read_csv(directory / REQUIRED[1])
    sample = pd.read_csv(directory / REQUIRED[3])
    pd.read_csv(directory / REQUIRED[2])
    assert train.drop(columns='Annual Turnover').columns.tolist() == test.columns.tolist()
    assert len(test) == 500 and train['Registration Number'].is_unique and test['Registration Number'].is_unique
    assert set(train['Registration Number']).isdisjoint(test['Registration Number'])
    assert sample.columns.tolist() == ['Registration Number', 'Annual Turnover']
    assert sample['Registration Number'].equals(test['Registration Number'])
    y = train['Annual Turnover'].to_numpy(float)
    assert np.isfinite(y).all() and (y > 0).all()
    return train, test, y


def feature_frame(raw):
    # Same deterministic cleaned representation already screened in v5.
    return ref.v5_features(raw, 'clean')


def splits(train, scheme):
    if str(scheme) == 'city':
        groups = feature_frame(train)['City']
        result = list(GroupKFold(5).split(train, groups=groups))
        for a, b in result:
            assert set(groups.iloc[a]).isdisjoint(groups.iloc[b])
    else:
        result = list(KFold(5, shuffle=True, random_state=int(scheme)).split(train))
    counts = np.zeros(len(train), int)
    for a, b in result:
        assert not set(a).intersection(b)
        counts[b] += 1
    assert (counts == 1).all()
    return result


def native_frame(X, categories=None):
    result = X.copy()
    if categories is None:
        categories = {c: sorted(X[c].unique().tolist())
                      for c in X.select_dtypes(include=['object', 'string']).columns}
    for c, values in categories.items():
        result[c] = pd.Categorical(result[c], categories=values, ordered=False)
    return result, categories


def lgb_parameters(spec, seed, trees):
    return dict(n_estimators=trees, learning_rate=.03, num_leaves=spec['leaves'],
                min_child_samples=50, reg_lambda=20, reg_alpha=.1,
                colsample_bytree=.8, subsample=.8, subsample_freq=1,
                cat_smooth=20, min_data_per_group=20, objective=spec['objective'],
                alpha=.9, metric='rmse', random_state=seed, n_jobs=2, verbosity=-1,
                deterministic=True, force_col_wise=True)


def cat_parameters(seed, depth=5):
    return dict(iterations=850, learning_rate=.04, depth=depth, l2_leaf_reg=10,
                bootstrap_type='Bernoulli', subsample=.7, random_seed=seed,
                thread_count=2, verbose=False, allow_writing_files=False)


def domain_weights(X, test_X, seed):
    """Training-row probabilities are cross-fitted; outer validation is never passed."""
    both = pd.concat([X, test_X], ignore_index=True)
    domain = np.r_[np.zeros(len(X), int), np.ones(len(test_X), int)]
    probabilities = np.zeros(len(both))
    categorical = both.select_dtypes(include=['object', 'string']).columns.tolist()
    for fold, (a, b) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(both, domain)):
        model = CatBoostClassifier(iterations=300, depth=3, learning_rate=.035,
                                   l2_leaf_reg=20, loss_function='Logloss',
                                   random_seed=seed + fold, thread_count=2,
                                   verbose=False, allow_writing_files=False)
        model.fit(both.iloc[a], domain[a], cat_features=categorical)
        probabilities[b] = model.predict_proba(both.iloc[b])[:, 1]
    p = np.clip(probabilities[:len(X)], 1e-6, 1 - 1e-6)
    ratio = p / (1 - p) * len(X) / len(test_X)
    clipped = np.clip(ratio, .5, 2.)
    weights = clipped / clipped.mean()
    return weights, {'ratio_clip': [.5, 2.], 'normalized_min': float(weights.min()),
                     'normalized_max': float(weights.max()),
                     'effective_sample_size': float(weights.sum() ** 2 / (weights ** 2).sum()),
                     'mean_weight': float(weights.mean()), 'domain_seed': seed}


def fit_component(raw, y, test_covariates, spec, seed):
    X = feature_frame(raw)
    target = np.asarray(y, float) / 1e6
    categorical = X.select_dtypes(include=['object', 'string']).columns.tolist()
    diagnostics = {'seed': seed, 'train_rows': len(X)}
    if spec['kind'] == 'lgb':
        a, b = train_test_split(np.arange(len(X)), test_size=.2, random_state=seed + 301)
        inner_X, inner_categories = native_frame(X.iloc[a])
        inner_valid, _ = native_frame(X.iloc[b], inner_categories)
        pilot = LGBMRegressor(**lgb_parameters(spec, seed, 2000))
        pilot.fit(inner_X, target[a], eval_set=[(inner_valid, target[b])],
                  eval_metric='rmse', callbacks=[early_stopping(100, first_metric_only=True, verbose=False)])
        trees = max(1, pilot.best_iteration_)
        native_X, categories = native_frame(X)
        model = LGBMRegressor(**lgb_parameters(spec, seed, trees))
        model.fit(native_X, target)
        diagnostics['trees'] = trees
        return {'model': model, 'categories': categories, 'kind': 'lgb', 'diagnostics': diagnostics}
    if spec['kind'] == 'sqrt':
        transformed = np.sqrt(target)
        inner_prediction = np.zeros(len(X))
        for fold, (a, b) in enumerate(KFold(3, shuffle=True, random_state=seed + 317).split(X)):
            model = CatBoostRegressor(**cat_parameters(seed + 100 + fold, 6), loss_function='RMSE')
            model.fit(X.iloc[a], transformed[a], cat_features=categorical)
            inner_prediction[b] = np.maximum(model.predict(X.iloc[b]), 0) ** 2
        correction = float(np.mean(target - inner_prediction))
        model = CatBoostRegressor(**cat_parameters(seed, 6), loss_function='RMSE')
        model.fit(X, transformed, cat_features=categorical)
        diagnostics.update(correction_millions=correction, correction_source='three-fold inner OOF')
        return {'model': model, 'correction': correction, 'kind': 'sqrt', 'diagnostics': diagnostics}
    assert spec['kind'] == 'weighted'
    weights, info = domain_weights(X, feature_frame(test_covariates), seed + 911)
    model = CatBoostRegressor(**cat_parameters(seed), loss_function='Huber:delta=40')
    model.fit(X, target, cat_features=categorical, sample_weight=weights)
    diagnostics.update(info)
    return {'model': model, 'kind': 'weighted', 'diagnostics': diagnostics}


def predict_component(fitted, raw):
    X = feature_frame(raw)
    if fitted['kind'] == 'lgb':
        X, _ = native_frame(X, fitted['categories'])
    result = fitted['model'].predict(X)
    if fitted['kind'] == 'sqrt':
        result = np.maximum(result, 0) ** 2 + fitted['correction']
    return np.maximum(result, 0) * 1e6


def component_oof(train, test, y, spec, scheme, progress=None):
    individual = np.zeros((len(y), len(SEEDS)))
    diagnostics = []
    for fold, (a, b) in enumerate(splits(train, scheme)):
        for j, seed in enumerate(SEEDS):
            fitted = fit_component(train.iloc[a], y[a], test, spec, seed + fold)
            individual[b, j] = predict_component(fitted, train.iloc[b])
            diagnostics.append({'fold': fold, **fitted['diagnostics']})
        if progress:
            progress(f'fold {fold + 1}/5 complete')
    return individual.mean(axis=1), individual, diagnostics


def fit_final(train, test, y, choice, contract):
    fitted = {'choice': choice, 'contract': contract, 'new': [], 'baseline': None}
    prediction = np.zeros(len(test))
    if choice['weight'] < 1:
        _, baseline, legacy = ref.full_fit(train, test, y, contract)
        fitted['baseline'] = legacy
        prediction += (1 - choice['weight']) * baseline
    if choice['weight']:
        for seed in SEEDS:
            model = fit_component(train, y, test, SPECS[choice['name']], seed)
            fitted['new'].append(model)
            prediction += choice['weight'] * predict_component(model, test) / len(SEEDS)
    return prediction, fitted


def predict_final(fitted, raw):
    choice = fitted['choice']
    p = np.zeros(len(raw))
    if choice['weight'] < 1:
        p += (1 - choice['weight']) * ref.predict_final(raw, fitted['baseline'], fitted['contract'])
    if choice['weight']:
        p += choice['weight'] * np.mean([predict_component(m, raw) for m in fitted['new']], axis=0)
    return p

In [ ]:
CONTRACT = json.loads('{"v2_configs": {"engineered": {"kind": "cat", "variant": "engineered", "depth": 6, "iterations": 260}, "engineered_drop": {"kind": "cat", "variant": "engineered_drop", "depth": 6, "iterations": 216}, "ordered_depth5": {"kind": "cat", "variant": "base", "depth": 5, "boosting_type": "Ordered", "iterations": 219}, "huber20_engineered": {"kind": "cat", "variant": "engineered_drop", "depth": 5, "loss": "Huber:delta=20000000", "iterations": 699}, "huber40": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 664}, "clip120": {"kind": "cat", "variant": "base", "depth": 5, "cap": 120000000.0, "iterations": 591}, "spline_ridge": {"kind": "spline", "variant": "base", "alpha": 100}, "svr_core": {"kind": "svr", "variant": "core", "C": 20}, "huber40_engineered": {"kind": "cat", "variant": "engineered", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 619}, "huber40_monotone": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "monotone": true, "iterations": 447}}, "v2_weights": {"engineered": 0.06448907249408362, "engineered_drop": 0.09631834163153197, "ordered_depth5": 0.08507039177778712, "huber20_engineered": 0.06274452861282048, "huber40": 0.08855362607985372, "clip120": 0.07794631558814363, "spline_ridge": 0.15648185431084338, "svr_core": 0.058428294374292714, "huber40_engineered": 0.09327747966602483, "huber40_monotone": 0.2166900954646185}, "v3_parameters": {"variant": "base", "depth": 5, "iterations": 850, "delta": 40, "subsample": 0.7, "l2": 10}, "v3_seeds": [1042, 2042, 3042], "version": "v5", "v3_weight": 0.25, "seeds": [10242, 20242], "components": {"sqrt_d6_correct1": {"spec": {"power": 0.5, "depth": 6, "correct_mean": true}, "weight": 0.25}, "subspace_h40": {"spec": {"loss": "Huber:delta=40", "rsm": 0.5, "iterations": 1200}, "weight": 0.25}, "quantile0.6": {"spec": {"loss": "Quantile:alpha=0.6", "depth": 5}, "weight": 0.25}}, "development": {"settings": 22, "shortlisted_settings": 6, "splits": [42, 71], "selection": {"name": "top3", "new_weight": 0.75, "rmse42": 19817236.327581998, "rmse71": 19840250.361659143, "mean_rmse": 19828743.34462057, "gain42": 21791.213259298354, "gain71": 29833.246005315334}, "screening_is_optimistic": true}, "confirmation": {"new_splits": [18421, 18422], "previous_splits": [2026, 303], "city_stress": true, "all_splits_reuse_previously_seen_labels": true}}')
CHOICE = json.loads('{"name": "v5", "family": "v5", "weight": 0.0}')

train, test, y = load_data(DATA_DIR)
ref.get_splits = splits
assert np.isclose(ref.rmse([0, 3], [0, 0]), np.sqrt(4.5))
changed = train.copy()
changed['Annual Turnover'] = 0
changed['Registration Number'] = -999
pd.testing.assert_frame_equal(feature_frame(train), feature_frame(changed))
assert feature_frame(train).columns.equals(feature_frame(test).columns)
print('Train:', train.shape, 'Test:', test.shape)
print('PASS: metric, target/ID exclusion, matching feature schema and input alignment.')
print('Chosen configuration:', json.dumps(CHOICE, indent=2))

## Recorded development and locked confirmation

These are the actual completed experiment results, embedded as records for review.
The next section independently retrains one complete five-fold comparison without
using prediction caches. The development assignments are 42/71; confirmation uses
18421/18422 and cleaned city groups. All assignments reuse previously examined labels.

In [ ]:
development_table = pd.DataFrame(json.loads('[{"name":"sqrt_crossfit","family":"sqrt_crossfit","weight":0.25,"rmse_42":19812630.98579677566886,"gain_42":4605.341785222291946,"rmse_71":19841051.035211097449064,"gain_71":-800.673551950603724,"mean_rmse":19826841.010503932833672},{"name":"sqrt_crossfit","family":"sqrt_crossfit","weight":0.5,"rmse_42":19814778.87403629720211,"gain_42":2457.453545700758696,"rmse_71":19848831.171313848346472,"gain_71":-8580.809654705226421,"mean_rmse":19831805.022675074636936},{"name":"domain_huber","family":"domain_huber","weight":0.25,"rmse_42":19829774.87563306838274,"gain_42":-12538.548051070421934,"rmse_71":19842780.240425746887922,"gain_71":-2529.878766603767872,"mean_rmse":19836277.558029409497976},{"name":"native_l2_15","family":"native_l2","weight":0.25,"rmse_42":19827598.265000652521849,"gain_42":-10361.937418654559224,"rmse_71":19859161.172185990959406,"gain_71":-18910.810526847839356,"mean_rmse":19843379.718593321740627},{"name":"sqrt_crossfit","family":"sqrt_crossfit","weight":0.75,"rmse_42":19823677.797170665115118,"gain_42":-6441.469588667154312,"rmse_71":19863582.568855065852404,"gain_71":-23332.207195922732353,"mean_rmse":19843630.183012865483761},{"name":"domain_huber","family":"domain_huber","weight":0.5,"rmse_42":19848097.568215470761061,"gain_42":-30861.240633472800255,"rmse_71":19850974.015954531729221,"gain_71":-10723.654295388609171,"mean_rmse":19849535.792084999382496},{"name":"native_l2_7","family":"native_l2","weight":0.25,"rmse_42":19829627.183205548673868,"gain_42":-12390.855623550711243,"rmse_71":19874000.342710580676794,"gain_71":-33749.981051437556744,"mean_rmse":19851813.762958064675331},{"name":"sqrt_crossfit","family":"sqrt_crossfit","weight":1.0,"rmse_42":19839318.670675180852413,"gain_42":-22082.343093182891607,"rmse_71":19885289.71348063647747,"gain_71":-45039.35182149335742,"mean_rmse":19862304.192077908664942},{"name":"domain_huber","family":"domain_huber","weight":0.75,"rmse_42":19872188.405924819409847,"gain_42":-54952.078342821449041,"rmse_71":19864824.679571941494942,"gain_71":-24574.317912798374891,"mean_rmse":19868506.542748380452394},{"name":"native_huber_15","family":"native_huber","weight":0.25,"rmse_42":19876581.241818081587553,"gain_42":-59344.914236079901457,"rmse_71":19892562.020485538989305,"gain_71":-52311.658826395869255,"mean_rmse":19884571.631151810288429},{"name":"native_l2_15","family":"native_l2","weight":0.5,"rmse_42":19868220.74454303830862,"gain_42":-50984.416961044073105,"rmse_71":19907215.87575875222683,"gain_71":-66965.514099609106779,"mean_rmse":19887718.310150898993015},{"name":"domain_huber","family":"domain_huber","weight":1.0,"rmse_42":19902026.442244630306959,"gain_42":-84790.114662632346153,"rmse_71":19884320.410160258412361,"gain_71":-44070.048501115292311,"mean_rmse":19893173.426202446222305},{"name":"native_huber_7","family":"native_huber","weight":0.25,"rmse_42":19890793.194807283580303,"gain_42":-73556.867225285619497,"rmse_71":19907670.720165278762579,"gain_71":-67420.358506135642529,"mean_rmse":19899231.957486279308796},{"name":"native_l2_7","family":"native_l2","weight":0.5,"rmse_42":19872196.122340846806765,"gain_42":-54959.794758848845959,"rmse_71":19937464.732785359025002,"gain_71":-97214.371126215904951,"mean_rmse":19904830.427563101053238},{"name":"native_l2_15","family":"native_l2","weight":0.75,"rmse_42":19938918.813469707965851,"gain_42":-121682.485887710005045,"rmse_71":19984204.232266064733267,"gain_71":-143953.870606921613216,"mean_rmse":19961561.522867888212204},{"name":"native_huber_15","family":"native_huber","weight":0.5,"rmse_42":19980162.717858277261257,"gain_42":-162926.390276279300451,"rmse_71":19987924.578500512987375,"gain_71":-147674.216841369867325,"mean_rmse":19984043.648179396986961},{"name":"native_l2_7","family":"native_l2","weight":0.75,"rmse_42":19944749.914772007614374,"gain_42":-127513.587190009639016,"rmse_71":20030361.091610781848431,"gain_71":-190110.72995163872838,"mean_rmse":19987555.503191396594048},{"name":"native_huber_7","family":"native_huber","weight":0.5,"rmse_42":20015377.851419657468796,"gain_42":-198141.52383765950799,"rmse_71":20024905.330286178737879,"gain_71":-184654.968627035588724,"mean_rmse":20020141.590852916240692},{"name":"native_l2_15","family":"native_l2","weight":1.0,"rmse_42":20039374.158066775649786,"gain_42":-222137.83048477768898,"rmse_71":20089793.604749608784914,"gain_71":-249543.243090465693967,"mean_rmse":20064583.88140819221735},{"name":"native_l2_7","family":"native_l2","weight":1.0,"rmse_42":20046963.00038943067193,"gain_42":-229726.672807432711124,"rmse_71":20152282.403981577605009,"gain_71":-312032.042322430759668,"mean_rmse":20099622.70218550413847},{"name":"native_huber_15","family":"native_huber","weight":0.75,"rmse_42":20127297.80103962123394,"gain_42":-310061.473457623273134,"rmse_71":20125726.075453758239746,"gain_71":-285475.713794615119696,"mean_rmse":20126511.938246689736843},{"name":"native_huber_7","family":"native_huber","weight":0.75,"rmse_42":20190045.703582808375359,"gain_42":-372809.376000810414553,"rmse_71":20191086.507825907319784,"gain_71":-350836.146166764199734,"mean_rmse":20190566.105704359710217},{"name":"native_huber_15","family":"native_huber","weight":1.0,"rmse_42":20317040.273659832775593,"gain_42":-499803.946077834814787,"rmse_71":20305102.488389145582914,"gain_71":-464852.126730002462864,"mean_rmse":20311071.381024487316608},{"name":"native_huber_7","family":"native_huber","weight":1.0,"rmse_42":20413511.183811020106077,"gain_42":-596274.856229022145271,"rmse_71":20405018.40580602735281,"gain_71":-564768.044146884232759,"mean_rmse":20409264.794808518141508}]'))
confirmation_table = pd.DataFrame(json.loads('[]'))
DECISION = json.loads('{"decision": "retain portal-scored v5", "choice": {"name": "v5", "family": "v5", "weight": 0.0}, "candidates": [], "qualified": [], "lock_sha256": "e3dd56f7f67a2c48353b33a3d774ae500fbdfa94f543f1f98dc23450d6e1ae1a", "limitation": "All splits reuse previously examined labels; portal accuracy remains unmeasured."}')

best_by_family = development_table.sort_values('mean_rmse').groupby('family', sort=False).head(1)
print('Best development setting per family (positive gain is better):')
print(best_by_family[['name', 'weight', 'gain_42', 'gain_71']].to_string(index=False))
print()
print('Locked confirmation:')
print(confirmation_table.to_string(index=False) if len(confirmation_table) else 'No candidate passed development.')
print()
print('Decision:', DECISION['decision'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
positions = np.arange(len(best_by_family))
axes[0].barh(positions - .16, best_by_family['gain_42'] / 1000, height=.3, label='Seed 42', color='#2563eb')
axes[0].barh(positions + .16, best_by_family['gain_71'] / 1000, height=.3, label='Seed 71', color='#0f766e')
axes[0].set_yticks(positions, best_by_family['name'])
axes[0].axvline(0, color='#334155')
axes[0].set(xlabel='RMSE improvement over v5 (INR thousands)', title='Recorded development comparisons')
axes[0].legend()
axes[1].hist(y / 1e6, bins=45, color='#2563eb', edgecolor='white')
axes[1].set(xlabel='Annual turnover (INR millions)', ylabel='Restaurants', yscale='log', title='All original labels remain in scoring')
fig.tight_layout()
plt.show()

## Fresh five-fold reproduction

This cell trains every fold of assignment 18421 again, including the frozen v5
reference and the selected new component when applicable. The compressed reference
arrays below are used **only after fitting to check reproduction**; they never supply
model predictions. The assertion requires maximum differences below INR 0.01.

In [ ]:
EXPECTED_BASELINE_OOF = np.frombuffer(zlib.decompress(base64.b64decode('...')), dtype='<f8').copy()
EXPECTED_SELECTED_OOF = np.frombuffer(zlib.decompress(base64.b64decode('...')), dtype='<f8').copy()

fresh_baseline = ref.evaluate(train, y, 18421, CONTRACT)['candidate']
assert np.max(np.abs(fresh_baseline - EXPECTED_BASELINE_OOF)) < .01
if CHOICE['weight']:
    fresh_component, _, fresh_diagnostics = component_oof(
        train, test, y, SPECS[CHOICE['name']], 18421, print)
    fresh_selected = (1 - CHOICE['weight']) * fresh_baseline + CHOICE['weight'] * fresh_component
else:
    fresh_selected = fresh_baseline.copy()
assert np.max(np.abs(fresh_selected - EXPECTED_SELECTED_OOF)) < .01
print('PASS: freshly trained complete five-fold predictions reproduce the independent reference within INR 0.01.')
fresh_validation_rows = []
for segment, mask in [('all', np.ones(len(y), bool)), ('<=40M', y <= 40e6),
                      ('40-80M', (y > 40e6) & (y <= 80e6)), ('>80M', y > 80e6)]:
    fresh_validation_rows.append({'segment': segment, 'rows': int(mask.sum()),
        'v5_RMSE': ref.rmse(y[mask], fresh_baseline[mask]),
        'selected_RMSE': ref.rmse(y[mask], fresh_selected[mask])})
print(pd.DataFrame(fresh_validation_rows).to_string(index=False))

## Fresh full fit, behavioral checks and CSV export

All selected models are fitted again on the official training data. Expected test
predictions are numerical reproduction references, not hidden target labels.
The final file keeps the original 500 restaurant IDs in their original order.

In [ ]:
def behavior_checks(fitted, test, prediction):
    permutation = np.random.default_rng(91).permutation(len(test))
    assert np.max(np.abs(predict_final(fitted, test.iloc[permutation]) - prediction[permutation])) < .01
    edge = test.iloc[:3].copy()
    edge['City'] = 'never_seen_city_v6'
    edge['Cuisine'] = pd.NA
    edge['Facebook Popularity Quotient'] = np.nan
    edge['Overall Restaurant Rating'] = np.nan
    assert np.isfinite(predict_final(fitted, edge)).all()
    altered = test.copy()
    altered['Registration Number'] = -777
    altered['Annual Turnover'] = 1e30
    assert np.max(np.abs(predict_final(fitted, altered) - prediction)) < .01
    submission = pd.DataFrame({'Registration Number': test['Registration Number'], 'Annual Turnover': prediction})
    ref.check_submission(submission, test)
    invalid = [submission.iloc[:-1], pd.concat([submission, submission.iloc[:1]], ignore_index=True),
               submission.iloc[::-1].reset_index(drop=True), submission.iloc[:, ::-1],
               submission.rename(columns={'Annual Turnover': 'Turnover'}), submission.assign(index=0)]
    for column, value in [('Registration Number', 'submission'), ('Registration Number', test['Registration Number'].iloc[1]),
                          ('Annual Turnover', np.nan), ('Annual Turnover', np.inf), ('Annual Turnover', -np.inf), ('Annual Turnover', -1.)]:
        bad = submission.copy()
        if isinstance(value, str):
            bad[column] = bad[column].astype(object)
        bad.loc[0, column] = value
        invalid.append(bad)
    for bad in invalid:
        try:
            ref.check_submission(bad, test)
        except (AssertionError, ValueError, TypeError):
            continue
        raise AssertionError('A malformed submission was accepted')
    return {'row_order': True, 'unique_ids': True, 'finite_nonnegative': True,
            'missing_unseen_values': True, 'target_and_id_invariance': True,
            'permutation_invariance': True, 'malformed_cases_rejected': len(invalid),
            'rows': len(test), 'columns': submission.columns.tolist()}

In [ ]:
EXPECTED_FULL_FIT = np.frombuffer(zlib.decompress(base64.b64decode('eJwNl2k41VsUxguZ0nAzE1e3FFEplXQjL0kiSlLUVYjCwZnn4X/OMZUGhMhQUVRSIXOlkSKlEpWZroqbClEU93zaH/azn732Xu9612819xxpvrlWgMTtrl/6+yQgVwZMfnLlYVH01cQ1RCTOHUlRGJ4bDQOW9ZHL7WLscVtnIOgQwaGtyuzCXhGGBCUdN2bzMaGtmpe7iw2j8TdaClo8ZHabXHayJbDNOnWJjaLsnFDBdn49AxFNXstS/6NCp460KT1eCk2ytd3RxxL8EZD2sWQkFOcCC0wNN4uxXq8pRDWKh/yCnJihs2T0nFY0XXdYALdrja0Gd1j41U1SOVjGwGiHbd3THAm+XnRfvkgzHG0H5W1p1hKk5yZ8y48VI/epeXsOUwDqcre8g450XCVttPHVjITvzpy6cGMJTuR5/gi7xoWhYoya5lkxjh/o0T9WREdLLzujwVwADcdHP9S8pch2H+zIW0NggKqe57GYDmHuk/DXf4sQVPvxUUEfFz8T1VcQs3nIGXkZuN+dgwDFzLVVehwEHehfzauSYFPW6/xacyq2TnsYEsDmYK3Id7T1vRgWfj26oXNZeJYwfDy4VIBbxZEBcQpSxNpvIBm0RuLL6fYLFYksfC+S66bTODjJvHompFyCDV9nc1Jk//8g6fAQbQYdRbWWzU8fCnF4wwIdepAIT1vXEPVxfOQXXvNKvkGBuW905ZSxEJ9nJV65clSEtsCCtHu50Ugquv/zoo4Q4yazSoOeEEgcXekHXRo8VP6JsWsNBX8PXbt8gg2dAEXuwaoo8G6tWnFXyoST5RydhN8CBCwou2B6lA0rk/G+LF0GpK8fJmrwOViql3WqX6Yr/+ZceedZQjzcQFr2Z6MIC78Mh3mfEGCv8i63tB1sTM2dkfFiMYF27wjOyB4Ctx5F+lc94aJJm3im6iuG1ZUbXwbpfIS3cNekN/KxLEFHedtGDhaNns8KsRDj8cu65Lt8CXLfOex7fFv2fqnfzGUZPPhlD/XNlobDpjY/a/03FtJnSX/rSSLQrXi3/L6CLO+deT9tDAWw67NmV/pyYT0Ss2yhtyzu2gp4JIgwdXzvVlIQC/tva8+vkghwpmCF8Y9eApdJcYP1rQQGT8fwRgVMXH+f1Rh2UIrKHZdYPV6RcD703uBHKRf+nffEqi8EuKH9Q6F3iRA1jjsHTvjRsNmwed/1AQISncw5FBsJxCo5po9UGVgzapw1X1WCrtX7/G7VitG8+UGXgmY07iaws5WpFCwNi4s8xpJiOPWXFyWPge27zNeUxEfj6vUPaiNxFCxvSBsRVArgFBtQdf4kH0dCUx9UaJBxOv1D2tmGGBh9rDBIVuCC0+UUSl3OQnmkzcNeOypO6rikc9JEkOPaynVvlUBJS/Ii5TwDr8IvtzEYTAz/47FqRR4XsTMarmc+ImD9tl4pmSuE57W0ihQ1FnpvPsfATwJPOn6Ysuv5+O+ZZOFfSmJU1jod7FXhIrvTo5SQiGGUNrfwXqYA49+qHXJk+uvOXvQ4MIQOaV7/ruRJNk5znvZ8YfMR+tpf/ntbJERv7EkcCR9O0fM/2BgL0Bbka/DbhAsebf+Nyi9iLBxepLrflopprwamvdlAQ6XcgvuC4zzM14/46qMqRvial8n918gIoG8rK73IhyBwc9LdYjFU8rXsdv6W5XnKwu2JjQiTVjc9tPJEkNjyP36YxsXZV1FvEhIkKF96wYPOj8arE680Lv1DoMpR7d/ZhyUQuse5Rvmz8blnwrFGjwILjx3DrzkSGJb6dl+U4+DYmTMpg/1RsEur1jUzIRC9ZUhn9xgTdYy1ZeRSAqnji+/26fCQtmDFliNvCSTxn8wLHBBgrs3LEHILC9xhS/LOJgaWU6u/tsv2V1Nme2eqRoFe5h+pbEUgpX9hsns6C3mPCk8sVBfhneruKJ9xLlYnNTzv1mPC5E5SCMWbgsPpDEGABRUP+uYcLWYRKPdseU3KjYTF1J7IU7soyOjZZbxAFp/N7G/k6AkODKQ7zVc7i0He5zZW3s9Gl/nvOnd1Jh6pprr9OiCEuVeb+s9fNPgEfxiKSBYhRT4lo89Min5VruKdGhES3SJnmKsR2J2tH1ChxkSNT117DsFBbFZZzSwDGpod9DZMfZbgl0bdx1/yMn+y4/18L/OFgpddSYtpQpQseXm88QYDS8cLspvmMDCRNPrJtUGAliLf6osqEpAyTPzUTGmYfP8pn98pWzsspZ5eZPjXVNj0LBBjVujm75p+kYgfmd52mhoJXp+uwtMtYoRsj5k8a8dG9POQU/ddgtExHKvsVMqAkDaoNSVloYTarGH5lxTVFRYtT5342Gg2fv2OvRjlhN+zb4k83N7aqD4xnUDdoetuyzuE8FPP/GhzjINsIf/L7wQxwnrP2rZnSLD7Oe38qJIEYw6ZOzzniBBaKRUtMOJjWHVhnbqrFMTPF+qXdPgolztxSF/MQZ299ylPv2gcSv5103ExBZ8MWYWOT9hYEpr6r+ncSJitm/Chy/retq/dicXTRBD9kzAW+JyBfzcXdr5JDsfjiOk6aREiVCXoLpuoYaB96tmjJJle7rrFrz/wUoxzT9buLFOKwkhwdX9xDhNdAy9VvxeIkORmkkBfyQbZfDjm+X0u3tgsL/gym4ZH01epTiURCCnZlb+2IBr7a4+p3Sqkg984q+BsDYG4gemUB5pk/Pn5VHaSUiT+iFfq/vaDhusUPf8MBhcxOmcJd5l+/5Dy2sjVHKy7IHhpvZ4Jz78z7rlAiPIg+8/as0W4sCXNZ+CTBA0b9N13rhIg9YHXX8cHmNCqP7q0U8YNW5UUby7xIMCtC16jESGBlmF0uNZPCr47PzGgGhJQDK9+Q1iJYEWq4Ny4LMWtYUcJ6yoDWQt636dMIxDlxLuk9p2JOTQB27qfA1eNiHe3V4ci9K/N26atJbD93DR7/m0q5nu0HHfoYkPX/n36TTIHk5kRX5pPRqL36L29uVMSBD+zrDHSEiOK4uB8qJAL6rJOqxYSgWtuGrdLiwX4Y/CHu4uQhRbl36EhM8nIlcshtWWyYGJcc85uuhTSyTH9bAU+Tl6ydbZ+TcHY+YfRr2T1++L7p5kNBixMPB+Mf2HGxIQSzdIxko+HPU+VyCV0vOqvUltuLEbBxv7TPsoMvFbeHTJ2hwnfZcXnlkpoeKX//Xv/Yxq+WX0ddmoQw+5YCkv/mRRvkw126jozEXwkvoJziIl5cYZ5F7poSN1AnvCmMXHzGt+w5y0Hhy3/7LI7xoXxvJUcs78pUNbx6Z01JYa/XoFLZyAB/bXnhWF3eDDbsvBjSrkUq3JytA3NWKjsvUQlVkpwam+ZUrCAjXL91evifpHx6malbkcADQc1ejdvUBHhQNtjqzyZn9+svVNceVGEF0fYVEfLSNDnhI8d4USjSXPxYnsSGyeytItGN/IwNJJzLumATK95yp7KTnTYF9/xmbcnGovyV1hVu4nRYTSt8GuqBFtffbgd70ZB21af7zMMxfBxkS9SceXAqEe866qirH4OvWvs2yhFvcpeZVexBPf0rxve7yHgeH6o/Y+rMs6sWmz1S8REwq6qnkulfJjXHlIP82LhLGfHf+3fZD4d/mbd0s8sNJHqNj7oYuDEUYv/PgcxYV0YWfzMMAYzxhYdUN4vhXzgvv7uGQR0smfMSdrJhUAC/kUHDux03j1xJ/HxdOJx++R+DjYPNDrUJ8v6YJ2frXMmBTqN1X87hXNwPjwgwqaWAb1kFlEs4zk5uRRXgXc0WmerWLqNyervzkyXTX/HgBHbMiOmTgL9q//gjax+ikcM7nmRGDB1Njic+54HrWkXFthV0UC5xNHYuFSI8xNG6zoLZT60b812/XEeuq26lMhzuTjs/XXai8BoBDM/63hX0aHNWJstTRKh+t/WdIfjR+CbrGH5tYoKwb6wlT9X8eBKeZ2a/oCJqJnaORs0qXhn+aEo6zEfg0lnHHgJx2T91PYasYmGecafmWFTUTiSMHetxyYmxMlvlydPhmCRs/OLJTcleNaf3n/fmQ/3rWrznNoFcA0V5vvMI3DQf0mByzU+ll6X6L8+yoA56bhNgZEAc4bCxl748REy4qHlW0bgUG6wt6knHQeNj02q/RTjUsR8YdyOKGzfa+Da5kpGs7TZ491DBn552vtYFovw6KzzQEowFfrjvvfTXBgIW3FjsT9JCNUf9d1iLSb6xa1zJ7vZaIwebRoIYKJyiJlxkCSbC05xb58z4SD88gPT0loOGFNTW/r2UKH30zc+o1OIt785Xl6OMm6pWVW35K0YXsEjVrfMCMhxJs0v2wtBvhWb0uhIQ1uFZ3mRBx9EYFx0mBsboRnHYn2axOBovni33EQIql6VotEHMq5zsvwHyWx0m+p+3LeIiedeZp/eMSWoDslYOeQv4+7jWo7b4oWY3hzTnZoSiYPrCxMUgyT4sVjrMG1MCtcfE2or7CWYpGknNwfRoXYF3XvsWAjprXXx3MnH73B5XeZsKrZvjfEfdRAg5Yt03cKNLFh/1Bs6v5mFGY1v/aPSqLAWGm83kfHomdMB2m5/RqHDrWtZsTsfd/1D9WxVZPxp8VpdQYmKV5sDpvKXM+CqbaF4rZuH1dK90xt0o2G+Z72rNJSEK+fiTffKyTh0r0PLPgkPG0/bmHKW8KA50bJQ9wwXtz6s7LLolHFufWMR5RAFA3PuBop7xTA5nfswdqbsfs0oy0fqUqwb79zjvUKKo9et5bTrwvEEJWaeIWRUxOaLSRpSPAgbOvSOwcOKPxkmqVlcCKuUAkczCMwXGT04dVcKM8GaJXHnqKiLHdy3Tp5AjZybx22VIAx+Musf2sJH77iCHdtSxn/9OxbNn5SiNOuNd6JsHjFQrViRmSfG7peLbL+dJFCpVdZQn0igWxBB/70nBsrMvqFdYTy08m1Ui9JDYSi/R7DNlQJyU0m9NIMPesmVKsZ6Dkr8Ll5qbGUhvul44eNdAvQWHUtI0uLDr/CT++kaDp7NWLnp37NcyM/7ajv3PwE2TNfVd5ZxQpmxS2a2gA+SWbmaGgiIm7IsEmQc81ZpXbGHCxf/A9SCQ5w=')), dtype='<f8').copy()

test_prediction, fitted = fit_final(train, test, y, CHOICE, CONTRACT)
maximum_difference = float(np.max(np.abs(test_prediction - EXPECTED_FULL_FIT)))
assert maximum_difference < .01, maximum_difference
checks = behavior_checks(fitted, test, test_prediction)
output_path = OUTPUT_DIR / 'submission.csv'
submission = pd.DataFrame({'Registration Number': test['Registration Number'], 'Annual Turnover': test_prediction})
submission.to_csv(output_path, index=False)
roundtrip = pd.read_csv(output_path)
ref.check_submission(roundtrip, test)
assert np.max(np.abs(roundtrip['Annual Turnover'].to_numpy() - test_prediction)) < .01
assert len(output_path.read_text().splitlines()) == 501
print('PASS: fresh full fit reproduces the independent export; maximum difference INR', maximum_difference)
print('Behavior checks:', json.dumps(checks, indent=2))
print('CSV:', output_path.name)
print('SHA256:', hashlib.sha256(output_path.read_bytes()).hexdigest())
print(submission.head().to_string(index=False))
print(submission['Annual Turnover'].describe().to_string())

## Interpretation and portal use

Lower RMSE is better. These are robustness comparisons on reused training labels,
not an estimate of a new portal score. The portal's scoring subset, remaining
submission allowance and deadline timezone still require checking in the portal.
No account access or upload is performed by this notebook.

Use at most two qualified portal trials. Record the exact CSV hash and returned
RMSE, retaining the lowest observed score. If the decision is **retain portal-scored
v5**, the exported model reproduces the existing scored solution; do not spend a
portal trial uploading it again. A large prediction change does not establish accuracy.

Native categories: https://lightgbm.readthedocs.io/en/stable/Advanced-Topics.html

Validation: https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html

In [ ]:
try:
    from google.colab import files
except ImportError:
    print('Local output saved:', output_path)
else:
    files.download(str(output_path))